In [ ]:
import numpy as np
import pandas as pd
import pickle
import sys

sys.path.append('..')
sys.path.append('../src')

from src.text_features import clean_text, decode_labels, TFIDFVectorizer, truncate_text
from src.logistic_regression import LogisticRegression

In [ ]:
df_subm = pd.read_csv('subm1.csv', sep=";")

texts = [clean_text(truncate_text(t)) for t in df_subm['Text'].tolist()]

with open('../vectorizers/vectorizer_logreg.pkl', 'rb') as f:
    vec = pickle.load(f)

X_subm = vec.transform(texts)
X_subm_dense = X_subm.toarray() if not isinstance(X_subm, np.ndarray) else X_subm

print(f"Dados transformados (shape: {X_subm_dense.shape})")

In [ ]:
model = LogisticRegression(n_classes=5)

model.load('../models/model_logreg.npz')

preds_idx = [model.predict(x) for x in X_subm_dense]

preds_raw = decode_labels(preds_idx)

In [ ]:
label_mapping = {
    'human': 'Human',
    'anthropic': 'Anthropic',
    'google': 'Google',
    'openai': 'OpenAI',
    'meta': 'Meta'
}

preds_formatadas = [label_mapping.get(label.lower(), label) for label in preds_raw]
df_subm['Labels'] = preds_formatadas

print(df_subm.head())
print(df_subm['Labels'].value_counts())

filename = 'subm1-g2-MEI-A.csv' 

df_subm.to_csv(filename,index=False)

print(f"\nFicheiro '{filename}' guardado com sucesso!")